In [ ]:
                                    # Capstone Project 3 (Smart Marketing Prediction System)(ML Pipeline Project)

In [ ]:
# Scenario
# A fast-growing e-commerce company called ShopEasy is struggling with inefficient marketing campaigns.
# Every day thousands of users visit their website. The marketing team spends a large amount of money showing ads, discounts, 
# and promotional emails, but they don't know which customers are actually likely to buy something.
# Currently:
#     Many customers browse but never purchase
#     Marketing money is wasted on the wrong users
#     The company wants to predict purchase probability
# The data science team has been asked to build a machine learning system that predicts whether a customer will purchase a product during a session.
# If the system predicts high probability of purchase, the system will:
#     show personalized product recommendations
#     offer targeted discounts
#     prioritize marketing campaigns
# If the system predicts low probability, the company will avoid spending marketing resources.
# However, the dataset contains both numerical and categorical features, so the data science team must design a complete ML pipeline.
# Build a machine learning model that predicts whether a user will purchase (1) or not purchase (0) during a website session.
# The model must be implemented using scikit-learn pipelines, including:
#     Encoding techniques
#     Feature preprocessing
#     Model training
#     Model selection
#     Hyperparameter tuning)

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [7]:
df = pd.read_excel('DatasetCapstoneProject3.xlsx')

In [8]:
df

,CustomerID,Age,Gender,Device,Traffic_Source,Time_on_Website,Pages_Visited,Ad_Clicks,Previous_Purchases,Purchased
0,1,23,Male,Mobile,Social Media,5,3,1,0,0
1,2,35,Female,Desktop,Search Engine,12,8,3,2,1
2,3,29,Male,Tablet,Social Media,8,5,2,1,0
3,4,41,Female,Mobile,Email Campaign,15,10,4,3,1
4,5,22,Female,Desktop,Direct,4,2,0,0,0
5,6,38,Male,Desktop,Search Engine,18,11,5,4,1
6,7,30,Male,Mobile,Social Media,7,4,1,1,0
7,8,27,Female,Tablet,Email Campaign,9,6,2,2,1
8,9,45,Male,Desktop,Direct,20,12,5,5,1
9,10,24,Female,Mobile,Social Media,6,3,1,0,0


In [9]:
X = df.drop('Purchased', axis=1)
y = df['Purchased']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
num_features = ['Age', 'Time_on_Website', 'Pages_Visited', 'Ad_Clicks', 'Previous_Purchases']
cate_features = ['Gender', 'Device', 'Traffic_Source']

In [14]:
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])
cate_pipeline = Pipeline([
    ('encode', OneHotEncoder(handle_unknown="ignore"))
])

In [16]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cate', cate_pipeline, cate_features)
])
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [18]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cate', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
y_pred = pipeline.predict(X_test)

In [20]:
print("\nModel Accuracy")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


Model Accuracy
1.0

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


Confusion Matrix
[[2 0]
 [0 3]]


In [25]:
y_prob = pipeline.predict_proba(X_test)
print(y_prob[:5])

[[0.90332294 0.09667706]
 [0.22769464 0.77230536]
 [0.95146624 0.04853376]
 [0.01754572 0.98245428]
 [0.01038206 0.98961794]]


In [28]:
param_grid = {
    'model__C' : [0.01, 0.1, 1, 10],
    'model__solver': ["lbfgs", "liblinear"]
}
grid = GridSearchCV(
    pipeline, 
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

D:\anaconda\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__solver': ['lbfgs', 'liblinear']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cate', ...)]"


In [29]:
print("\nBest Parameters:")
print(grid.best_params_)
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print("\nBest Model Accuracy:")
print(accuracy_score(y_test, y_pred_best))


Best Parameters:
{'model__C': 1, 'model__solver': 'lbfgs'}

Best Model Accuracy:
1.0
